## Использование
- Выполни настройку окружения и убедись, что заданы `LLM_API_KEY` и `RETRIEVA_KEY`.
- При необходимости переопредели промпты в блоке `PROMPT_OVERRIDES`.
- Прогоняй тестовые примеры для каждого агента и фиксируй наблюдения/регрессии.
- В конце сессии не забудь закрыть HTTP-клиент (последняя ячейка).

## 1. Настройка окружения
Ниже вставлен минимальный пролог для корректной работы импортов и асинхронных вызовов в Jupyter. Если ноутбук открыт не из корня репозитория — скрипт поднимется на уровень вверх автоматически.

In [1]:
import os
import sys
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ai_assistant").exists() and PROJECT_ROOT.parent.exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "ai_assistant").exists():
        PROJECT_ROOT = candidate

ASSISTANT_PATH = PROJECT_ROOT / "ai_assistant"
if str(ASSISTANT_PATH) not in sys.path:
    sys.path.append(str(ASSISTANT_PATH))

load_dotenv()
nest_asyncio.apply()
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/akayo/Desktop/Хакатоны/alpha


## 2. PROMPT_OVERRIDES
Этот блок позволяет временно переопределять любые поля из `agent_system.service.prompts.Prompts`.
Укажи новый текст в словаре ниже (оставь `None`, если менять ничего не нужно).

In [2]:
from ai_assistant.agent_system.service.prompts import Prompts

PROMPT_OVERRIDES = {
    "ALIGNMENT": None,
    "ANSWER_SYSTEM": None,
    "ANSWER_USER_TEMPLATE": None,
    "RAG_VALIDATOR_SYSTEM": None,
    "RAG_VALIDATOR_USER_TEMPLATE": None,
    "INTENT_SYSTEM": None,
    "INTENT_USER_TEMPLATE": None,
    "VALIDATOR_SYSTEM": None,
    "VALIDATOR_USER_TEMPLATE": None,
}


def apply_prompt_overrides(overrides: dict) -> list:
    applied = []
    for name, text in overrides.items():
        if not text:
            continue
        if not hasattr(Prompts, name):
            raise AttributeError(f"Unknown prompt field: {name}")
        cleaned = text.strip()
        setattr(Prompts, name, cleaned)
        applied.append(name)
    return applied

APPLIED_PROMPTS = apply_prompt_overrides(PROMPT_OVERRIDES)
if APPLIED_PROMPTS:
    print("Переопределены промпты:", ", ".join(APPLIED_PROMPTS))
else:
    print("Переопределений нет — используются значения по умолчанию.")


Переопределений нет — используются значения по умолчанию.


## 3. Инициализация общих зависимостей
Эта ячейка создаёт клиента LLM, контроллер поиска по базе знаний и экземпляры всех агентов. Убедись, что ключи заданы до запуска: можно либо положить их в `.env`, либо временно прописать прямо в переменные ниже.

In [1]:
import os
from web.config import InferenceConfig
from ai_assistant.agent_system.rag import RetrieverController
from ai_assistant.agent_system.utils.llm_client import AsyncLLMClient
from ai_assistant.agent_system.agents.intent_rewrite import IntentRewriteAgent
from ai_assistant.agent_system.agents.retrieval import RetrievalAgent
from ai_assistant.agent_system.agents.rag_validator import RagValidatorAgent
from ai_assistant.agent_system.agents.answer import AnswerAgent
from ai_assistant.agent_system.agents.validator import ValidationAgent
from ai_assistant.agent_system.agents.web_searcher import WebSearcherAgent
from ai_assistant.agent_system.service.pipeline import AgentPipeline

LLM_API_KEY = os.getenv("LLM_API_KEY") or "PASTE_KEY_HERE"
if not LLM_API_KEY or LLM_API_KEY == "PASTE_KEY_HERE":
    raise RuntimeError("Задай переменную окружения LLM_API_KEY перед тестами")


llm_client = AsyncLLMClient(
    api_key=LLM_API_KEY,
    base_url=InferenceConfig.OPENROUTER_BASE,
    model=InferenceConfig.OPENROUTER_MODEL,
    timeout=InferenceConfig.TIMEOUT,
    verify=False,
)

# retriever = RetrieverController()
intent_agent = IntentRewriteAgent(llm_client)
# retrieval_agent = RetrievalAgent(retriever)
answer_agent = AnswerAgent(llm_client)
rag_validator_agent = RagValidatorAgent(llm_client)
validator_agent = ValidationAgent(llm_client)
web_search_agent = WebSearcherAgent(llm_client)
# pipeline = AgentPipeline(llm_for_agents=llm_client, retriever=retriever)

print("LLM model:", llm_client.model)

LLM model: deepseek/deepseek-chat-v3.1


## 4. IntentRewriteAgent
Тестовые примеры ниже показывают, как агент классифицирует намерения и формирует запросы для дальнейших шагов. Меняй список `intent_samples` и анализируй JSON, который вернёт модель.

In [ ]:
intent_samples = [
    "Привет! Нужны идеи, как поднять продажи в WB.",
    "Спасибо, вопросов больше нет",
    "Какие штрафы за просрочку поставки FBO?",
]

async def inspect_intents(samples):
    results = []
    for text in samples:
        parsed = await intent_agent.run(text)
        results.append((text, parsed))
    return results

intent_results = await inspect_intents(intent_samples)
for text, parsed in intent_results:
    print("────" * 12)
    print("User:", text)
    print("Agent output:", parsed)

────────────────────────────────────────────────
User: Привет! Нужны идеи, как поднять продажи в WB.
Agent output: {'exit': False, 'kb_query': 'идеи для увеличения продаж на Wildberries', 'answer_query': 'Пользователь ищет идеи для увеличения продаж на платформе Wildberries'}
────────────────────────────────────────────────
User: Спасибо, вопросов больше нет
Agent output: {'exit': True}
────────────────────────────────────────────────
User: Какие штрафы за просрочку поставки FBO?
Agent output: {'exit': False, 'kb_query': 'штрафы за просрочку поставки FBO', 'answer_query': 'Пользователь интересуется штрафами за просрочку поставки FBO'}


## 5. RetrievalAgent
Укажи искомую тему, чтобы проверить, какие документы поднимет ретривер и подходят ли они под новый промпт. Эти данные можно затем напрямую передать AnswerAgent.

In [ ]:
kb_query = "логистика FBO"
kb_items = retrieval_agent.run(kb_query)
print(f"Получено {len(kb_items)} фрагментов")
for item in kb_items:
    print("────" * 12)
    print(f"ID: {item.get('id')} | score={item.get('score')}")
    print("TITLE:", item.get("title"))
    print("TEXT PREVIEW:", (item.get("knowledge") or "")[:400], "...")

[2025-11-27 18:43:38,814] INFO: Retrieval request succeeded (status=200) | extra={"taskName": "Task-11"}
[2025-11-27 18:43:38,817] INFO: Найдено 5 релевантных фрагментов. | extra={"taskName": "Task-11"}


Получено 5 фрагментов
────────────────────────────────────────────────
ID: 1 | score=0.841953456401825
TITLE: "Особенности схемы FBO на маркетплейсе Wildberries"
TEXT PREVIEW: Обработанный_текст:

FBO (fulfillment by operator) на Wildberries — это система, помогающая продавцам (селлерам) с упаковкой, маркировкой и доставкой товаров. Это позволяет предпринимателям сосредоточиться на более важных задачах. Wildberries выступает как маркетплейс, где товары размещаются и продвигаются. Платформа предлагает полный цикл услуг для успешных продаж. ...
────────────────────────────────────────────────
ID: 2 | score=0.8368889093399048
TITLE: "Преимущества и недостатки схемы FBO на Wildberries"
TEXT PREVIEW: Обработанный текст:

Система FBO на Wildberries: плюсы и минусы

Плюсы:
- Освобождение от рутинных задач: нет необходимости в обработке и упаковке заказов.
- Готовая логистика и хранение: можно сразу приступить к торговле.
- Свободное время для развития бизнеса и отдыха.
- Быстрая доставка из с

## 6. AnswerAgent
Используй результаты поиска (или задай свои `sample_kb`) и наблюдай, как меняется черновик после правок промпта.

In [ ]:
answer_query = "Как подготовить поставку на FBO?"
fallback_kb = [
    {
        "id": 0,
        "title": "Placeholder",
        "knowledge": "Уточните тему, чтобы ответ построился на фактах.",
        "score": 0.0,
    }
]

sample_kb = kb_items or fallback_kb

draft_answer = await answer_agent.run(answer_query, sample_kb)
print(draft_answer)

Чтобы подготовить поставку на FBO на Wildberries, учтите следующие шаги:

1. **Определите тип поставки:** Убедитесь, что вы выбираете подходящий способ отправки в зависимости от объема и типа товара, будь то коробками или паллетами.

2. **Подготовьте упаковку:** Обеспечьте качественную упаковку, так как товар должен оставаться в исправном состоянии до момента доставки покупателю.

3. **Выбор склада:** Чтобы оптимизировать затраты, особенно если бюджет ограничен, рекомендуется использовать склады в Москве или Казани для удобной доставки товаров по России.

4. **Планируйте продажи:** Анализируйте спрос, чтобы избежать затоваривания и эффективнее управлять запасами.

Это поможет вам обеспечить плавное управление поставками и концентрацию на развитии бизнеса.


## 7. ValidationAgent
Проверь, как валидатор шлифует черновик: достаточно ли персонализации и соблюдены ли ограничения по стилю.

In [ ]:
draft_for_validation = draft_answer or "Передай сюда черновик, если AnswerAgent возвращает пустую строку."
final_answer_preview = await validator_agent.run(draft_for_validation)
print(final_answer_preview)

Чтобы успешно организовать поставку на FBO на Wildberries, обратите внимание на несколько ключевых шагов: 

1. **Определите тип поставки:** Выберите способ отправки, который лучше всего подходит для вашего товара, будь то коробки или паллеты. 

2. **Подготовьте упаковку:** Заботьтесь о качественной упаковке, чтобы ваш товар оставался в идеальном состоянии до доставки покупателю. 

3. **Выбор склада:** Для оптимизации затрат, особенно если ваш бюджет ограничен, рассмотрите использование складов в Москве или Казани для удобной доставки по России. 

4. **Планируйте продажи:** Анализируйте спрос, чтобы избежать затоваривания и эффективно управлять запасами. 

Следуя этим шагам, вы сможете лучше управлять поставками и сосредоточиться на развитии вашего бизнеса.


## 8. RagValidatorAgent
Здесь можно проверить, как фильтрация KB влияет на итоговый ответ и требуется ли fallback к веб-поиску перед генерацией черновика.

In [ ]:
rag_validator_query = "Какие штрафы за просрочку поставки FBO?"
if 'kb_items' not in globals():
    raise RuntimeError("Сначала запусти блок RetrievalAgent, чтобы получить тестовые документы.")

validated_kb_items = await rag_validator_agent.run(rag_validator_query, kb_items)
print(f"Вопрос: {rag_validator_query}")
print(f"Исходных фрагментов: {len(kb_items)}")
print(f"После фильтра: {len(validated_kb_items)}")
for item in validated_kb_items:
    print('────' * 12)
    print(f"ID: {item.get('id')} | score={item.get('score')}")
    print("TITLE:", item.get("title"))
    print("TEXT PREVIEW:", (item.get("knowledge") or "")[:400], "...")
if not validated_kb_items:
    print("RagValidatorAgent вернул пустой список — можно задействовать веб-поиск.")


Вопрос: Какие штрафы за просрочку поставки FBO?
Исходных фрагментов: 5
После фильтра: 5
────────────────────────────────────────────────
ID: 1 | score=0.841953456401825
TITLE: "Особенности схемы FBO на маркетплейсе Wildberries"
TEXT PREVIEW: Обработанный_текст:

FBO (fulfillment by operator) на Wildberries — это система, помогающая продавцам (селлерам) с упаковкой, маркировкой и доставкой товаров. Это позволяет предпринимателям сосредоточиться на более важных задачах. Wildberries выступает как маркетплейс, где товары размещаются и продвигаются. Платформа предлагает полный цикл услуг для успешных продаж. ...
────────────────────────────────────────────────
ID: 2 | score=0.8368889093399048
TITLE: "Преимущества и недостатки схемы FBO на Wildberries"
TEXT PREVIEW: Обработанный текст:

Система FBO на Wildberries: плюсы и минусы

Плюсы:
- Освобождение от рутинных задач: нет необходимости в обработке и упаковке заказов.
- Готовая логистика и хранение: можно сразу приступить к торговле.
- Своб

## 9. WebSearcherAgent
Fallback-агент помогает быстро получить свежие факты. Этот блок показывает, какие веб-фрагменты он возвращает и как выглядят ссылки.

In [3]:
web_search_query = "Регистрация ИП"
web_fragments = await web_search_agent.search(web_search_query)
print(f"Запрос: {web_search_query}")
print(f"Найдено фрагментов: {len(web_fragments)}")
for fragment in web_fragments:
    print('────' * 12)
    print(f"ID: {fragment.get('id')} | title={fragment.get('title')}")
    print((fragment.get("knowledge") or "")[:600], "...")
if not web_fragments:
    print("DuckDuckGo ничего не вернул — проверь запрос или лимиты API.")


[2025-12-06 14:46:14,930] INFO: Web search completed | title='Веб-поиск: Регистрация ИП' | knowledge='March 26, 2025 - Оформление ИП на себя в 2025 году через налог ру. Онлайн регистрация бизнеса, цены, выбор между банком и госуслугами, условия подачи заявления без личного посещения. Регистрация ИП са' | extra={"query": "Регистрация ИП", "task": "Ты веб-аналитик. Найди актуальные данные по запросу: \"Регистрация ИП\". Сначала используй поисковую систему, затем открой минимум 3 релевантных источников. Если DuckDuckGo вернул пустой результат, немедленно завершай работу и сообщи, что свежих данных нет. В финальной сводке перечисли резюме и добавь ссылки.", "tool": "DuckDuckGoSearchRun", "result_source": "direct-search", "fallback_triggered": true}


Запрос: Регистрация ИП
Найдено фрагментов: 1
────────────────────────────────────────────────
ID: web_0325ea54 | title=Веб-поиск: Регистрация ИП
March 26, 2025 - Оформление ИП на себя в 2025 году через налог ру. Онлайн регистрация бизнеса, цены, выбор между банком и госуслугами, условия подачи заявления без личного посещения. Регистрация ИП самостоятельно: пошаговая инструкция для начинающих. Варианты регистрации ИП, порядок подготовки документов, подводные камни при регистрации и многое другое. July 29, 2025 - Эта инструкция проведет вас по всему пути: от подготовки и выбора налогового режима до получения статуса ИП. Мы разберем все доступные способы подачи документов, сравним их плюсы и минусы и дадим краткий чек-лист первых действий ...


## 10. End-to-end AgentPipeline
Полный прогон через все стадии помогает убедиться, что связка промптов работает согласованно.

In [4]:
user_query = "Какие новости в ноябре 2025 есть от Вайлдберриз"
pipeline_result = await pipeline.run(user_text=user_query, user_id=999, history=[])
print("Exit flag:", pipeline_result.get("exit"))
print("KB hits:", len(pipeline_result.get("kb", [])))
print("Final answer:", pipeline_result.get("final"))

NameError: name 'pipeline' is not defined

## 11. Завершение сессии
После экспериментов корректно закрывай HTTP-сессию клиента LLM, чтобы не копились открытые соединения.

In [4]:
await llm_client.aclose()
print("LLM client is closed")

NameError: name 'llm_client' is not defined

# Отдельный агент для проверки каких-то гипотез